# Lab 4 · Error logging & parameterised backfill

Wrap steps so any failure is logged to `ops.etl_error_log` and re-raised, then **backfill** the missing May 2026 activity idempotently. A pipeline can override the date window at runtime.

> **Attach** the `lh_resident360` Lakehouse first.

In [ ]:
import datetime, json
import notebookutils
from pyspark.sql import Row
from delta.tables import DeltaTable
spark.sql("CREATE SCHEMA IF NOT EXISTS ops")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

## 1. Parameters — a pipeline can override these

In [ ]:
start_date = "2026-05-01"
end_date   = "2026-05-31"

## 2. Error-logging wrapper — failures land in `ops.etl_error_log`

In [ ]:
def run_step(step_name, fn):
    try:
        return fn()
    except Exception as e:
        err = spark.createDataFrame([Row(step=step_name, error=str(e)[:500],
                                         ts=datetime.datetime.now().isoformat())])
        err.write.mode("append").saveAsTable("ops.etl_error_log")
        raise

## 3. Backfill the missing May 2026 activity (idempotent MERGE)
The mirror is read-only, so recovered rows land in a Fabric table `bronze.activity_backfill`.

In [ ]:
import random
def backfill():
    s = datetime.date.fromisoformat(start_date)
    e = datetime.date.fromisoformat(end_date)
    resident_ids = [r.resident_id for r in
                    spark.table("hpb_databricks_mirror.gold.dim_resident").select("resident_id").collect()]
    rows = []
    for rid in resident_ids:
        rng = random.Random(hash(rid) & 0xffff)
        d = s
        while d <= e:
            steps = max(0, int(rng.gauss(6000, 2500)))
            rows.append(Row(resident_id=rid, activity_date=d, steps=steps,
                            mvpa_minutes=min(120, int(max(0, steps/300))),
                            sleep_minutes=int(rng.gauss(410, 60)), goal_met=1 if steps>=10000 else 0))
            d += datetime.timedelta(days=1)
    df = spark.createDataFrame(rows)
    if spark.catalog.tableExists("bronze.activity_backfill"):
        tgt = DeltaTable.forName(spark, "bronze.activity_backfill")
        (tgt.alias("t").merge(df.alias("s"),
            "t.resident_id = s.resident_id AND t.activity_date = s.activity_date")
           .whenNotMatchedInsertAll().execute())
    else:
        df.write.mode("overwrite").saveAsTable("bronze.activity_backfill")
    return df.count()

n = run_step("backfill_may_2026", backfill)
print(f"Backfilled {n} activity rows for {start_date}..{end_date}")
notebookutils.notebook.exit(json.dumps({"backfilled_rows": n, "window": f"{start_date}..{end_date}"}))